In [3]:
import pandas as pd
from dotenv import load_dotenv
import psycopg2
import os

In [4]:
load_dotenv()
url = os.getenv("DATABASE_URL")

In [5]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

query = """
SELECT timestamp, ticker, side, volume, price FROM orders
ORDER BY volume DESC
"""
cur.execute(query)
rows = cur.fetchall()

In [6]:
df = pd.DataFrame(rows, columns=['timestamp', 'ticker', 'side', 'volume', 'price'])
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)
df.head()

,ticker,side,volume,price
timestamp,,,,
2026-04-29 16:47:14+00:00,SR320CF6,SELL,270000,12
2026-04-01 10:56:46+00:00,SR310CD6B,BUY,153143,8
2026-03-26 15:18:50+00:00,SR320CP6A,SELL,74862,4
2026-03-25 11:46:13+00:00,SR310CD6,BUY,63350,12
2026-03-25 06:19:33+00:00,SR310CC6D,BUY,51150,9


In [7]:
df.describe()

,volume,price
count,9912.000000,9912.000000
mean,1582.101493,5.803370
std,5143.079989,7.397118
min,0.000000,0.000000
25%,22.000000,0.000000
50%,116.000000,3.000000
75%,816.000000,9.000000
max,270000.000000,55.000000


In [8]:
buy_df = df[df['side'] == "BUY"]
buy_df.head()

,ticker,side,volume,price
timestamp,,,,
2026-04-01 10:56:46+00:00,SR310CD6B,BUY,153143,8
2026-03-25 11:46:13+00:00,SR310CD6,BUY,63350,12
2026-03-25 06:19:33+00:00,SR310CC6D,BUY,51150,9
2026-04-24 19:06:47+00:00,SR320CE6A,BUY,48182,8
2026-04-17 13:44:19+00:00,SR310CF6,BUY,47000,23


In [9]:
top_10_buy_df = buy_df.head(10)
top_10_buy_df

,ticker,side,volume,price
timestamp,,,,
2026-04-01 10:56:46+00:00,SR310CD6B,BUY,153143,8
2026-03-25 11:46:13+00:00,SR310CD6,BUY,63350,12
2026-03-25 06:19:33+00:00,SR310CC6D,BUY,51150,9
2026-04-24 19:06:47+00:00,SR320CE6A,BUY,48182,8
2026-04-17 13:44:19+00:00,SR310CF6,BUY,47000,23
2026-04-23 12:26:07+00:00,SR320CE6A,BUY,43566,10
2026-04-22 13:37:36+00:00,SR310CD6E,BUY,37584,15
2026-03-27 11:12:23+00:00,SR310CD6,BUY,37505,9
2026-04-17 06:56:10+00:00,SR320CD6E,BUY,36740,6


In [15]:
query = """
SELECT timestamp, ticker, bids, asks FROM orderbooks
WHERE ticker = ANY(%s)
"""
tickers = top_10_buy_df['ticker'].unique().tolist()
cur.execute(query, (tickers,))
rows = cur.fetchall()

In [20]:
orderbooks_df = pd.DataFrame(rows, columns=['timestamp', 'ticker', 'bids', 'asks'])
orderbooks_df['timestamp'] = pd.to_datetime(orderbooks_df['timestamp'])
orderbooks_df.set_index('timestamp', inplace=True)
orderbooks_df.head()

,ticker,bids,asks
timestamp,,,
2026-04-20 11:35:43+00:00,SR310CD6E,"[{'price': 14.91, 'quantity': 800}, {'price': ...","[{'price': 15.21, 'quantity': 200}, {'price': ..."
2026-04-20 11:35:43+00:00,SR320CD6E,"[{'price': 6.2, 'quantity': 362}, {'price': 6....","[{'price': 7.16, 'quantity': 5500}, {'price': ..."
2026-04-20 11:36:01+00:00,SR310CF6,"[{'price': 23.07, 'quantity': 500}, {'price': ...","[{'price': 24.94, 'quantity': 500}, {'price': ..."
2026-04-20 11:36:04+00:00,SR310CD6E,"[{'price': 14.91, 'quantity': 800}, {'price': ...","[{'price': 15.21, 'quantity': 200}, {'price': ..."
2026-04-20 11:36:04+00:00,SR310CF6,"[{'price': 23.27, 'quantity': 500}, {'price': ...","[{'price': 24.94, 'quantity': 500}, {'price': ..."


In [46]:
orderbooks_df['best_bid'] = orderbooks_df['bids'].apply(lambda bids: bids[0]['price'] if bids else None)
orderbooks_df['best_ask'] = orderbooks_df['asks'].apply(lambda asks: asks[0]['price'] if asks else None)
orderbooks_df['mid_price'] = orderbooks_df['best_bid'] + (orderbooks_df['best_ask'] - orderbooks_df['best_bid'])/2
prices_df = orderbooks_df[['ticker', 'mid_price']]
prices_df.head()

,ticker,mid_price
timestamp,,
2026-04-20 11:35:43+00:00,SR310CD6E,15.060
2026-04-20 11:35:43+00:00,SR320CD6E,6.680
2026-04-20 11:36:01+00:00,SR310CF6,24.005
2026-04-20 11:36:04+00:00,SR310CD6E,15.060
2026-04-20 11:36:04+00:00,SR310CF6,24.105


In [47]:
top_10_buy_df = top_10_buy_df.sort_index()
prices_df = prices_df.sort_values(['ticker', 'timestamp'])

In [48]:
base = pd.merge_asof(
    top_10_buy_df.sort_index(),
    prices_df.sort_values('timestamp'),
    left_on='timestamp',
    right_on='timestamp',
    by='ticker',
    direction='backward'
).rename(columns={'mid_price': 'mid_0'})

In [50]:
horizons = ['5min', '1h', '1D']

base = base.reset_index(drop=True)

prices_sorted = prices_df.sort_values('timestamp')

for h in horizons:
    tmp = base.copy()

    tmp['target_time'] = tmp['timestamp'] + pd.Timedelta(h)

    tmp = tmp.sort_values('target_time')

    tmp = pd.merge_asof(
        tmp,
        prices_sorted,
        left_on='target_time',
        right_on='timestamp',
        by='ticker',
        direction='forward'
    )

    base[f'mid_{h}'] = tmp['mid_price'].values

In [52]:
for h in horizons:
    base[f'ret_{h}'] = (base[f'mid_{h}'] - base['price']) / base['price']
base.head()

,ticker,timestamp,side,volume,price,mid_0,mid_5min,mid_1h,mid_1D,ret_5min,ret_1h,ret_1D
0,SR310CC6D,2026-03-25 06:19:33+00:00,BUY,51150,9,8.810,8.805,7.945,NaN,-0.021667,-0.117222,NaN
1,SR310CD6,2026-03-25 11:46:13+00:00,BUY,63350,12,12.335,12.280,12.140,10.980,0.023333,0.011667,-0.085000
2,SR310CD6,2026-03-27 11:12:23+00:00,BUY,37505,9,9.630,9.850,9.855,9.560,0.094444,0.095000,0.062222
3,SR310CD6B,2026-04-01 10:56:46+00:00,BUY,153143,8,8.345,8.240,7.580,7.435,0.030000,-0.052500,-0.070625
4,SR320CD6E,2026-04-17 06:56:10+00:00,BUY,36740,6,6.685,5.765,7.040,7.720,-0.039167,0.173333,0.286667
